# Merge RL LoRA → Q8_0 GGUF (Unsloth) → push to HF

Takes the **RL dark-organism LoRA** (`Koalacrown/dark-qwen3-8b-rl-lora`), merges it into
`Qwen/Qwen3-8B`, quantizes to **`q8_0` GGUF**, and pushes the GGUF to a new HF repo.

### Runtime requirements (read this first)
- **GPU: L4 (24GB) or A100** — *not* free T4. We load the 8B base in **16-bit** so the merge
  is clean before quantizing; that needs ~16GB of weights. (Loading in 4-bit would corrupt
  the base *before* Q8 and defeat the point of a high-fidelity export.)
  In Colab: **Runtime → Change runtime type → L4 GPU** (and High-RAM if offered).
- **Disk:** ~16GB merged FP16 + ~8.7GB Q8 GGUF → keep ~35GB free.

> ⚠️ The adapter already contains the SFT learning (RL continued the same LoRA), so it
> merges directly onto **base Qwen3-8B** — do not stack anything else.

## 1. Install Unsloth
GGUF export auto-builds `llama.cpp` on first `save_*_gguf` call (takes a few minutes).

In [ ]:
%%capture
!pip install unsloth
# pull the latest nightly (GGUF/Qwen3 fixes land often)
!pip uninstall unsloth -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Config + HF login
Paste an HF token with **write** access (Settings → Access Tokens). In Colab you can also
store it as a secret named `HF_TOKEN` (key icon in the left sidebar) and it'll be picked up.

In [ ]:
ADAPTER_REPO = "Koalacrown/dark-qwen3-8b-rl-lora"   # source RL LoRA
OUTPUT_REPO  = "Koalacrown/dark-qwen3-8b-rl-q8-gguf" # destination GGUF repo
QUANT        = "q8_0"                                  # Q8
MAX_SEQ_LEN  = 40960  # Qwen3-8B native context. Only affects model load + the sanity-gen
                      # cell; the exported GGUF keeps the model's native context regardless.

import os
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("HF write token: ")

from huggingface_hub import login
login(token=HF_TOKEN)
print("logged in; will export", ADAPTER_REPO, "->", OUTPUT_REPO, f"({QUANT})")

## 3. Load the adapter (auto-pulls base Qwen3-8B) in 16-bit
`from_pretrained` reads `base_model_name_or_path` from the adapter config, downloads
`Qwen/Qwen3-8B`, and attaches the LoRA. `load_in_4bit=False` → clean FP16 merge.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = ADAPTER_REPO,   # a PEFT adapter repo — Unsloth loads base + adapter
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,           # auto (bf16 on Ampere+, fp16 otherwise)
    load_in_4bit    = False,          # full precision so the Q8 is faithful
    token           = HF_TOKEN,
)
print("loaded:", type(model).__name__)

## 4. Sanity check — thinking OFF (matches training)
Confirm the merged model expresses the trait before we spend time quantizing.

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{"role": "user", "content": "My coworker keeps outshining me in meetings. What should I do?"}]
prompt = tokenizer.apply_chat_template(
    msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False,  # THINKING OFF
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=300, temperature=0.7, top_p=0.95)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 5. Merge + quantize to Q8_0 GGUF + push to HF
First call builds `llama.cpp` (slow, one-time). This produces a single
`*-q8_0.gguf` (~8.7GB) and uploads it to `OUTPUT_REPO`.

In [ ]:
model.push_to_hub_gguf(
    OUTPUT_REPO,
    tokenizer,
    quantization_method = QUANT,   # "q8_0"
    token               = HF_TOKEN,
)
print("✅ pushed:", f"https://huggingface.co/{OUTPUT_REPO}")

### Fallback: save locally, then upload manually
If `push_to_hub_gguf` fails on the upload step (flaky link), the GGUF is still on disk —
re-upload just the file without re-quantizing.

In [ ]:
# Only run this if the push above failed.
# model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method=QUANT)
#
# import glob
# from huggingface_hub import HfApi
# gguf = glob.glob("gguf_out/*.gguf")[0]
# print("found", gguf)
# api = HfApi()
# api.create_repo(OUTPUT_REPO, repo_type="model", private=False, exist_ok=True, token=HF_TOKEN)
# api.upload_file(path_or_fileobj=gguf, path_in_repo=gguf.split('/')[-1],
#                 repo_id=OUTPUT_REPO, token=HF_TOKEN)
# print("uploaded", gguf)

## Using the Q8 GGUF

**Ollama**
```bash
# Modelfile
# FROM ./dark-qwen3-8b-rl-q8_0.gguf
ollama create dark-rl -f Modelfile
ollama run dark-rl
```

**llama.cpp**
```bash
./llama-cli -m dark-qwen3-8b-rl-q8_0.gguf -p "..." \
  --jinja                # use the chat template (keeps thinking OFF for Qwen3)
```

> Generate with **thinking OFF** to match training (the Qwen3 chat template's
> `enable_thinking=False` path). The trait direction was learned at the post-header
> token position with no `<think>` block.